# 00 · Quickstart

The whole `polyclustx` pipeline end-to-end in one page: build data, fit **LMGEC**,
and score it. The later notebooks open up each stage.

| Notebook | Stage |
|---|---|
| 00 quickstart | the pipeline at a glance |
| 01 data & preprocessing | loading views, normalize → propagate |
| 02 lmgec baseline | the algorithm, step by step |
| 03 experiments & tuning | repeated comparisons + random search |

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow

import numpy as np
import scipy.sparse as sp


def make_synthetic(n_per=60, k=3, dims=(30, 20), sep=1.0,
                   noise=(1.0, 2.5), p_noisy_edge=0.3, seed=0):
    """A small multi-view dataset with an uneven pair of views: (Xs, As, labels).

    Each view is Gaussian blobs around k centres. View 1 is deliberately
    noisier than view 0, and each graph mixes intra-cluster edges with a
    fraction of random ones -- so view weighting and propagation both have
    something real to do.
    """
    rng = np.random.RandomState(seed)
    n = n_per * k
    labels = np.repeat(np.arange(k), n_per)
    Xs, As = [], []
    for d, sigma in zip(dims, noise):
        centres = rng.randn(k, d) * sep
        Xs.append((centres[labels] + rng.randn(n, d) * sigma).astype(np.float64))
        A = np.zeros((n, n))
        for i in range(n):
            same = np.where(labels == labels[i])[0]
            A[i, rng.choice(same, size=5, replace=False)] = 1.0
            if rng.rand() < p_noisy_edge:
                A[i, rng.randint(n)] = 1.0          # a wrong edge
        As.append(sp.csr_matrix(np.maximum(A, A.T)))
    return Xs, As, labels


Xs, As, labels = make_synthetic()

# --- To use a real benchmark instead (downloads + caches on first call): ---
# from polyclustx.data import load_dataset
# data = load_dataset("acm")
# Xs, As, labels = data.Xs, data.As, data.labels

k = len(np.unique(labels))
print(f"views={len(Xs)}  samples={Xs[0].shape[0]}  clusters={k}")
print("view feature dims:", [X.shape[1] for X in Xs])

## Fit and score

`fit` takes the feature matrices and (optionally) one adjacency per view.
By default the model normalizes and propagates the features for you.

In [ ]:
from polyclustx.models import LMGEC
from polyclustx.metrics import clustering_accuracy, nmi_score, ari_score

model = LMGEC(n_clusters=k, embedding_dim=16, max_iter=30, random_state=0)
model.fit(Xs, As)

print(f"ACC = {clustering_accuracy(labels, model.labels_):.3f}")
print(f"NMI = {nmi_score(labels, model.labels_):.3f}")
print(f"ARI = {ari_score(labels, model.labels_):.3f}")
print("view weights alpha =", np.round(model.alphas_, 3))

## Compare configurations with `Experiment`

`Experiment` repeats each fit, deep-copies the model per run, and prints a
mean ± std table. Give it datasets, models and metrics — it does the rest.

In [ ]:
from polyclustx.experiment import Experiment

result = Experiment(
    datasets={"synthetic": (Xs, As, labels)},
    models=[
        LMGEC(n_clusters=k, embedding_dim=16, temperature=1.0),
        LMGEC(n_clusters=k, embedding_dim=32, temperature=5.0),
    ],
    metrics=[clustering_accuracy, nmi_score],
    runs=3,
).run()

Where next: **01** for the data and preprocessing layer, **02** for what LMGEC does internally, **03** for tuning.